In [2]:
import pandas as pd
import cloudscraper
from datetime import datetime, timedelta, date
from dateutil.relativedelta import relativedelta
import boto3
from io import BytesIO
import time
import random
import json

BUCKET_NAME = "afdal-idx-stock-data-s3"
METADATA_KEY = "metadata/etl_control.json"

In [ ]:
BUCKET_NAME = "afdal-idx-stock-data-s3"

s3 = boto3.client("s3")

response = s3.list_objects_v2(
    Bucket=BUCKET_NAME
)

for obj in response.get("Contents", []):
    print(obj["Key"])

In [ ]:
s3.delete_objects(
    Bucket= BUCKET_NAME,
    Delete={
        "Objects":[
            {"Key": "raw/idx/stock_data.parquet"}
        ]
    }
)

In [ ]:
#mirip delete recursive
s3 = boto3.resource("s3")

bucket = s3.Bucket(BUCKET_NAME)
bucket.objects.filter(
    Prefix="raw/idx/year=2026/month=09/"
).delete()


In [ ]:
def upload_to_s3(df, bucket_name, key):
    s3 = boto3.client("s3")
    buffer = BytesIO()
    df.to_parquet(buffer, index=False, engine="pyarrow")
    s3.put_object(
        Bucket=bucket_name,
        Key=key,
        Body=buffer.getvalue()
    )

# upload_to_s3(df, BUCKET_NAME, "raw/idx/stock_data.parquet")

In [ ]:
df_check = pd.read_parquet(f"s3://{BUCKET_NAME}/raw/idx/year=2026/month=08/data.parquet")

df_check.head()

In [3]:
def read_s3(bucket_name, key):
    s3 = boto3.client("s3")
    response = s3.get_object(
        Bucket = bucket_name,
        Key = key
    )

    buffer = BytesIO(response["Body"].read())
    return pd.read_parquet(buffer)

df_check = read_s3(BUCKET_NAME, "raw/idx/year=2026/month=08/data.parquet")
df_check.head()

,No,IDStockSummary,Date,StockCode,StockName,Remarks,Previous,OpenPrice,FirstTrade,High,...,TradebleShares,WeightForIndex,ForeignSell,ForeignBuy,DelistingDate,NonRegularVolume,NonRegularValue,NonRegularFrequency,persen,percentage
0,1,4065341,2026-08-03T00:00:00,AADI,Adaro Andalan Indonesia Tbk.,--MO1SD0F10000A121------------,9225.0,9300.0,9300.0,9300.0,...,7.786892e+09,1.498977e+09,2396100.0,503600.0,,147964.0,1.339833e+09,16.0,None,None
1,2,4065342,2026-08-03T00:00:00,AALI,Astra Agro Lestari Tbk.,--MO113E100000D232------------,6875.0,6900.0,6900.0,7175.0,...,1.924688e+09,3.903268e+08,1111300.0,738100.0,,0.0,0.000000e+00,0.0,None,None
2,3,4065343,2026-08-03T00:00:00,ABBA,Mahaka Media Tbk.,--U-4100000000E614-E---------X,30.0,0.0,0.0,0.0,...,3.935893e+09,1.298451e+09,0.0,0.0,,0.0,0.000000e+00,0.0,None,None
3,4,4065344,2026-08-03T00:00:00,ABDA,Asuransi Bina Dana Arta Tbk.,--UO2105000000G412------------,3500.0,0.0,0.0,3530.0,...,6.208067e+08,5.947328e+07,0.0,0.0,,0.0,0.000000e+00,0.0,None,None
4,5,4065345,2026-08-03T00:00:00,ABMM,ABM Investama Tbk.,--MO1135000000A121------------,2510.0,2550.0,2550.0,2580.0,...,2.753165e+09,4.129748e+08,14600.0,197600.0,,16200.0,4.357800e+07,1.0,None,None


In [98]:
df=df_check
print(df.shape)
print(df.dtypes)
print(df.columns.tolist())
print(df.head().to_string())

(18297, 32)
No                       int64
IDStockSummary           int64
Date                    object
StockCode               object
StockName               object
Remarks                 object
Previous               float64
OpenPrice              float64
FirstTrade             float64
High                   float64
Low                    float64
Close                  float64
Change                 float64
Volume                 float64
Value                  float64
Frequency              float64
IndexIndividual        float64
Offer                  float64
OfferVolume            float64
Bid                    float64
BidVolume              float64
ListedShares           float64
TradebleShares         float64
WeightForIndex         float64
ForeignSell            float64
ForeignBuy             float64
DelistingDate           object
NonRegularVolume       float64
NonRegularValue        float64
NonRegularFrequency    float64
persen                  object
percentage              obj

In [ ]:
def get_idx_data(start_date: date, end_date: date = None):

    if end_date is None:
        end_date = start_date

    scraper = cloudscraper.create_scraper()
    url = "https://www.idx.co.id/primary/TradingSummary/GetStockSummary"

    dates = pd.date_range(
        start=start_date,
        end=end_date
    )

    all_data = []

    for current_date in dates:

        params = {
            "length": 9999,
            "start": 0,
            "date": current_date.strftime("%Y-%m-%d")
        }

        # Retry maksimal 5 kali
        for attempt in range(5):

            try:
                response = scraper.get(
                    url,
                    params=params,
                    timeout=30
                )

                if response.status_code == 200:
                    data_json = response.json()

                    df_idx = pd.DataFrame(data_json["data"])

                    if not df_idx.empty:
                        all_data.append(df_idx)

                    print(
                        f"{current_date.date()} - "
                        f"OK - {len(df_idx)} rows"
                    )

                    break

                elif response.status_code in [429, 503]:

                    # Exponential backoff + random jitter
                    wait = min(60, 3 * (2 ** attempt))
                    wait += random.uniform(0, 2)

                    print(
                        f"{current_date.date()} - "
                        f"HTTP {response.status_code}. "
                        f"Retry dalam {wait:.1f} detik..."
                    )

                    time.sleep(wait)

                else:
                    print(
                        f"{current_date.date()} - "
                        f"Request gagal: {response.status_code}"
                    )
                    break

            except Exception as e:

                wait = min(60, 3 * (2 ** attempt))
                wait += random.uniform(0, 2)

                print(
                    f"{current_date.date()} - "
                    f"Error: {e}. "
                    f"Retry dalam {wait:.1f} detik..."
                )

                time.sleep(wait)

        # Jeda normal antar request
        wait = random.uniform(2, 4)
        time.sleep(wait)

    if all_data:
        df = pd.concat(all_data, ignore_index=True)
    else:
        df = pd.DataFrame()

    return df


In [ ]:
last_date = datetime.strptime('2026-09-06', '%Y-%m-%d').date()
today = datetime.strptime('2026-09-09', '%Y-%m-%d').date()
# today = datetime.now().date()

In [ ]:
def get_month_list(start_date, end_date):
    months = set()
    current_date = start_date
    while current_date <= end_date:
        year=current_date.year
        month=current_date.month
        year_month = year*100+month
        months.add(year_month)
        current_date+=timedelta(days=1)
    return list(months)

In [ ]:
months = set()
current_date = last_date
while current_date <= today:
    year=current_date.year
    month=current_date.month
    year_month = year*100+month
    months.add(year_month)
    current_date+=timedelta(days=1)

months = list(months)

In [ ]:
months = get_month_list(start_date, end_date)

In [ ]:
for i, month in enumerate(months):
    mm = f'{month%100:02d}'
    yyyy = f'{int(month/100)}'
    # print(f'{mm}-{yyyy},{i}')
    if i!=len(months)-1:
        start_date = datetime.strptime(f'{yyyy}-{mm}-01', '%Y-%m-%d').date()
        end_date = start_date + relativedelta(months=1) - relativedelta(days=1)
        df = get_idx_data(start_date,end_date)
        key = f'raw/idx/year={yyyy}/month={mm}/data.parquet'
        print(f'{key} - jumlah: {len(df)}')
        upload_to_s3(df, BUCKET_NAME, key)
    else:
        first_month_date = f'{yyyy}-{mm}-01'
        list_date = pd.date_range(first_month_date, today)
        for j in list_date:
            date = j.date()
            df = get_idx_data(date)
            key = f'raw/idx/year={yyyy}/month={mm}/daily/date={date}/data.parquet'
            print(f'{key} - jumlah: {len(df)}')
            if len(df)>0:
                upload_to_s3(df, BUCKET_NAME, key)

In [ ]:
def compact_daily(year,month):
    s3 = boto3.client("s3")
    prefix = f"raw/idx/year={year}/month={month}/daily" 

    response =s3.list_objects_v2(
        Bucket=BUCKET_NAME,
        Prefix=prefix
    )

    if "Contents" not in response:
        print("data daily tidak ditemukan")
    else:
        keys = [obj["Key"] for obj in response.get("Contents", []) if obj["Key"].endswith(".parquet")]
        if not keys:
            print("data daily tidak ditemukan")
        else:
            df_compact = []
            for key in keys:
                df_daily = read_s3(BUCKET_NAME, key)
                df_compact.append(df_daily)
            print(f'Compact done for {year}-{month}')

            return pd.concat(df_compact, ignore_index=True)

In [ ]:
def auto_compact(lastrun_prev, current):
    month_list = get_month_list(lastrun_prev, current)
    if len(month_list)>1:
        for year_month in month_list[:-1]:
            mm = f'{year_month%100:02d}'
            yyyy = f'{year_month//100}'
            # print(f'{yyyy}-{mm}')
            df = compact_daily(yyyy,mm)
            if df is not None and not df.empty:
                key = f"raw/idx/year={yyyy}/month={mm}/data.parquet"
                # print(key)
                upload_to_s3(df, BUCKET_NAME, key)
                # Next dibuatkan delete data dailynya 


a=datetime.strptime('2026-05-29', '%Y-%m-%d').date()
b=datetime.strptime('2026-09-10', '%Y-%m-%d').date()
auto_compact(a,b)

In [ ]:
compact_daily('2026','08')

In [ ]:
start_date = datetime.strptime(f'{yyyy}-{mm}-01', '%Y-%m-%d').date()
end_date = start_date + relativedelta(months=1) - relativedelta(days=1)

In [ ]:
df = get_idx_data('2026-09-09')

In [ ]:
df.head()

In [ ]:
etl_metadata = {
    "datalake_idx":{
        "etl_id":1,
        "last_run":None,
        "last_success":None,
        "is_active":0,
        "start_time":None,
        "end_time":None,
        "message":None
    }
}

# metadata_json = json.dumps(
#     etl_metadata,
#     indent=4
# )

In [49]:
def upload_json_to_s3(data, bucket_name, key):
    s3 = boto3.client("s3")

    s3.put_object(
        Bucket=bucket_name,
        Key=key,
        Body=json.dumps(data, indent=4),
        ContentType="application/json"
    )

In [50]:
upload_json_to_s3(etl_metadata, BUCKET_NAME, "metadata/etl_control.json")

In [51]:
def read_json_s3(bucket_name, key):
    s3 = boto3.client("s3")

    response = s3.get_object(
        Bucket = bucket_name,
        Key = key
    )

    metadata = json.loads(response["Body"].read().decode("utf-8"))
    return metadata

In [95]:
tbl_meta = read_json_s3(BUCKET_NAME, "metadata/etl_control.json")
tbl_meta

{'datalake_idx': {'etl_id': 1,
  'last_run': '2026-09-10T23:21:36.382355',
  'last_success': '2026-09-10T23:21:47.129811',
  'is_active': 1,
  'start_time': '2026-09-10T23:21:34.037585',
  'end_time': '2026-09-10T23:21:47.129811',
  'message': None}}

In [96]:
etl_name = "datalake_idx"
metadata_etl = read_json_s3(BUCKET_NAME, METADATA_KEY)[etl_name]
metadata_etl

{'etl_id': 1,
 'last_run': '2026-09-10T23:21:36.382355',
 'last_success': '2026-09-10T23:21:47.129811',
 'is_active': 1,
 'start_time': '2026-09-10T23:21:34.037585',
 'end_time': '2026-09-10T23:21:47.129811',
 'message': None}

In [93]:
etl_name = "datalake_idx"
metadata_etl = read_json_s3(BUCKET_NAME, METADATA_KEY)[etl_name]
metadata_etl

{'etl_id': 1,
 'last_run': '2026-09-10T17:27:23.342214',
 'last_success': '2026-09-10T17:27:32.266580',
 'is_active': 1,
 'start_time': '2026-09-10T17:27:22.261609',
 'end_time': '2026-09-10T17:27:32.266580',
 'message': None}

In [77]:
print(metadata_etl["last_success"] \
                    if metadata_etl["last_success"] is not None \
                    else (datetime.today()-timedelta(days=1)).date())

2026-09-09


In [86]:
f'{(datetime.today()-timedelta(days=1)).date().year:04d}'

'2026'

In [62]:
def update_metadata(bucket_name, key, etl_name, **kwargs):
    metadata = read_json_s3(
        bucket_name, key
    )

    if etl_name not in metadata:
        raise ValueError(
            f"ETL '{etl_name}' tidak ditemukan di metadata"
        )
    etl_metadata = metadata[etl_name]

    invalid_keys = set(kwargs) - set(etl_metadata)

    if invalid_keys:
        raise ValueError(
            f"Key tidak valid: {invalid_keys}"
        )

    etl_metadata.update(kwargs)

    upload_json_to_s3(metadata, bucket_name, key)

In [64]:
update_metadata(
    BUCKET_NAME,
    METADATA_KEY,
    "datalake_idx",
    is_active=1
)

In [90]:
print(datetime.today())

2026-09-10 17:19:22.798525


In [7]:

# %%
import logging
from io import BytesIO
from datetime import datetime, timedelta, date

import boto3
import numpy as np
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

BUCKET_NAME = "afdal-idx-stock-data-s3"


# ============================================================
# LOGGING
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)


# ============================================================
# SOURCE → CURATED COLUMN MAPPING
# ============================================================

STANDARDIZE_COLUMN_MAP = {
    "IDStockSummary": "stock_summary_id",
    "Date": "trade_date",
    "StockCode": "stock_code",
    "StockName": "stock_name",
    "Remarks": "remarks",
    "DelistingDate": "delisting_date",

    "Previous": "previous_price",
    "OpenPrice": "open_price",
    "FirstTrade": "first_trade_price",
    "High": "high_price",
    "Low": "low_price",
    "Close": "close_price",
    "Change": "price_change",

    "Volume": "volume",
    "Value": "trading_value",
    "Frequency": "frequency",

    "IndexIndividual": "individual_index",

    "Offer": "offer_price",
    "OfferVolume": "offer_volume",
    "Bid": "bid_price",
    "BidVolume": "bid_volume",

    "ListedShares": "listed_shares",
    "TradebleShares": "tradeable_shares",
    "WeightForIndex": "index_weight",

    "ForeignSell": "foreign_sell",
    "ForeignBuy": "foreign_buy",

    "NonRegularVolume": "non_regular_volume",
    "NonRegularValue": "non_regular_value",
    "NonRegularFrequency": "non_regular_frequency",
}


EXPECTED_SOURCE_COLUMNS = set(STANDARDIZE_COLUMN_MAP.keys())

EXPECTED_CURATED_COLUMNS = list(STANDARDIZE_COLUMN_MAP.values())


# ============================================================
# DATA TYPE CONTRACT
# ============================================================

INTEGER_COLUMNS = [
    "stock_summary_id",
    "volume",
    "frequency",
    "offer_volume",
    "bid_volume",
    "listed_shares",
    "tradeable_shares",
    "foreign_sell",
    "foreign_buy",
    "non_regular_volume",
    "non_regular_frequency",
]

FLOAT_COLUMNS = [
    "previous_price",
    "open_price",
    "first_trade_price",
    "high_price",
    "low_price",
    "close_price",
    "price_change",
    "trading_value",
    "individual_index",
    "offer_price",
    "bid_price",
    "index_weight",
    "non_regular_value",
]

STRING_COLUMNS = [
    "stock_code",
    "stock_name",
    "remarks",
]

DATE_COLUMNS = [
    "trade_date",
    "delisting_date",
]


# ============================================================
# S3
# ============================================================

def read_df_from_s3(bucket_name, key, s3_client=None):
    """
    Read a Parquet file from S3 into a pandas DataFrame.
    """

    if s3_client is None:
        s3_client = boto3.client("s3")

    response = s3_client.get_object(
        Bucket=bucket_name,
        Key=key
    )

    buffer = BytesIO(response["Body"].read())

    return pd.read_parquet(buffer)


# ============================================================
# LOAD RAW
# ============================================================

def get_month_list(start_date, end_date):
    """
    Return sorted YYYYMM values between start_date and end_date.
    """

    months = set()

    current_date = start_date

    while current_date <= end_date:
        year_month = current_date.year * 100 + current_date.month
        months.add(year_month)

        current_date += timedelta(days=1)

    return sorted(months)


def load_raw(start_date, end_date):
    """
    Load raw IDX data from S3.

    Historical completed months are stored as:
        raw/idx/year=YYYY/month=MM/data.parquet

    Current month is stored as daily files:
        raw/idx/year=YYYY/month=MM/daily/date=YYYY-MM-DD/data.parquet
    """

    months = get_month_list(start_date, end_date)

    s3 = boto3.client("s3")

    dfs = {}

    for i, year_month in enumerate(months):

        month = year_month % 100
        year = year_month // 100

        mm = f"{month:02d}"
        yyyy = f"{year:04d}"

        # ----------------------------------------------------
        # Completed month
        # ----------------------------------------------------

        if i != len(months) - 1:

            key = (
                f"raw/idx/"
                f"year={yyyy}/"
                f"month={mm}/"
                f"data.parquet"
            )

            logger.info(f"Loading raw: {key}")

            dfs[key] = read_df_from_s3(
                BUCKET_NAME,
                key,
                s3
            )

        # ----------------------------------------------------
        # Current month → daily files
        # ----------------------------------------------------

        else:

            if len(months) > 1:
                first_date = date(year, month, 1)
            else:
                first_date = start_date

            date_range = pd.date_range(
                first_date,
                end_date
            )

            for current_date in date_range:

                current_date = current_date.date()

                key = (
                    f"raw/idx/"
                    f"year={yyyy}/"
                    f"month={mm}/"
                    f"daily/"
                    f"date={current_date}/"
                    f"data.parquet"
                )

                logger.info(f"Loading raw: {key}")

                dfs[key] = read_df_from_s3(
                    BUCKET_NAME,
                    key,
                    s3
                )

    logger.info(f"Loaded {len(dfs)} raw files")

    return dfs


# ============================================================
# SOURCE SCHEMA VALIDATION
# ============================================================

def validate_source_schema(dataframe):
    """
    Validate the expected schema from IDX.

    Missing expected columns are fatal because downstream
    transformation depends on them.

    Unexpected columns are only warnings because the source
    may add new fields in the future.
    """

    actual_columns = set(dataframe.columns)

    missing_columns = EXPECTED_SOURCE_COLUMNS - actual_columns
    unexpected_columns = actual_columns - EXPECTED_SOURCE_COLUMNS

    if missing_columns:
        raise ValueError(
            "Source schema validation failed. "
            f"Missing columns: {sorted(missing_columns)}"
        )

    if unexpected_columns:
        logger.warning(
            "Unexpected source columns detected: "
            f"{sorted(unexpected_columns)}"
        )

    logger.info("Source schema validation passed")

    return dataframe


# ============================================================
# COLUMN STANDARDIZATION
# ============================================================

def rename_columns(dataframe):
    """
    Rename IDX source columns into the internal curated schema.
    """

    return dataframe.rename(
        columns=STANDARDIZE_COLUMN_MAP
    )


# ============================================================
# DATA TYPE CASTING
# ============================================================

def cast_numeric_column(dataframe, column, dtype):
    """
    Cast numeric column while detecting values that cannot
    be converted.

    Existing null values are allowed.
    Newly-created null values caused by failed conversion
    are treated as errors.
    """

    original = dataframe[column]

    original_null = original.isna()

    converted = pd.to_numeric(
        original,
        errors="coerce"
    )

    conversion_error_mask = (
        ~original_null &
        converted.isna()
    )

    conversion_error_count = conversion_error_mask.sum()

    if conversion_error_count > 0:

        bad_values = original[
            conversion_error_mask
        ].unique()

        raise ValueError(
            f"Datatype conversion failed for column "
            f"'{column}'. "
            f"{conversion_error_count} invalid values. "
            f"Examples: {bad_values[:5]}"
        )

    dataframe[column] = converted.astype(dtype)

    return dataframe


def cast_datatypes(dataframe):
    """
    Apply the curated data type contract.
    """

    # --------------------------------------------------------
    # Integer
    # --------------------------------------------------------

    for column in INTEGER_COLUMNS:

        dataframe = cast_numeric_column(
            dataframe,
            column,
            "Int64"
        )

    # --------------------------------------------------------
    # Float
    # --------------------------------------------------------

    for column in FLOAT_COLUMNS:

        dataframe = cast_numeric_column(
            dataframe,
            column,
            "float64"
        )

    # --------------------------------------------------------
    # String
    # --------------------------------------------------------

    for column in STRING_COLUMNS:

        dataframe[column] = (
            dataframe[column]
            .astype("string")
            .str.strip()
        )

    # --------------------------------------------------------
    # Date / datetime
    # --------------------------------------------------------

    for column in DATE_COLUMNS:

        dataframe[column] = pd.to_datetime(
            dataframe[column],
            errors="coerce"
        ).dt.normalize()

    logger.info("Datatype casting passed")

    return dataframe


# ============================================================
# DATA QUALITY CHECK
# ============================================================

def quality_check(dataframe):
    """
    Validate structural and basic domain-level data quality.

    Structural violations are fatal.

    Domain anomalies such as unusual OHLC values are warnings
    because IDX may contain special trading conditions.
    """

    errors = []

    # ========================================================
    # 1. Required columns
    # ========================================================

    required_columns = [
        "trade_date",
        "stock_code",
    ]

    for column in required_columns:

        if dataframe[column].isna().any():

            count = dataframe[column].isna().sum()

            errors.append(
                f"NULL found in required column "
                f"'{column}': {count} rows"
            )

    # ========================================================
    # 2. Required string values must not be empty
    # ========================================================

    empty_stock_code = (
        dataframe["stock_code"]
        .fillna("")
        .str.strip()
        .eq("")
    )

    if empty_stock_code.any():

        count = empty_stock_code.sum()

        errors.append(
            f"Empty stock_code: {count} rows"
        )

    # ========================================================
    # 3. Duplicate grain
    #
    # Grain:
    # one stock per trading date
    # ========================================================

    grain_columns = [
        "trade_date",
        "stock_code",
    ]

    duplicate_mask = dataframe.duplicated(
        subset=grain_columns,
        keep=False
    )

    if duplicate_mask.any():

        duplicate_count = duplicate_mask.sum()

        errors.append(
            f"Duplicate grain detected for "
            f"({', '.join(grain_columns)}): "
            f"{duplicate_count} rows"
        )

    # ========================================================
    # 4. Numeric non-negative validation
    #
    # We DO NOT modify the values.
    # We only detect violations.
    # ========================================================

    non_negative_columns = [
        "previous_price",
        "open_price",
        "first_trade_price",
        "high_price",
        "low_price",
        "close_price",
        "volume",
        "trading_value",
        "frequency",
        "offer_price",
        "offer_volume",
        "bid_price",
        "bid_volume",
        "listed_shares",
        "tradeable_shares",
        "foreign_sell",
        "foreign_buy",
        "non_regular_volume",
        "non_regular_value",
        "non_regular_frequency",
    ]

    for column in non_negative_columns:

        invalid_mask = dataframe[column] < 0

        if invalid_mask.any():

            count = invalid_mask.sum()

            errors.append(
                f"Negative values found in "
                f"'{column}': {count} rows"
            )

    # ========================================================
    # 5. OHLC consistency
    #
    # Warning only.
    # ========================================================

    ohlc_validations = {

        "High < Low":
            dataframe["high_price"] <
            dataframe["low_price"],

        "Open < Low":
            dataframe["open_price"] <
            dataframe["low_price"],

        "Open > High":
            dataframe["open_price"] >
            dataframe["high_price"],

        "Close < Low":
            dataframe["close_price"] <
            dataframe["low_price"],

        "Close > High":
            dataframe["close_price"] >
            dataframe["high_price"],
    }

    for rule, invalid_mask in ohlc_validations.items():

        count = invalid_mask.sum()

        if count > 0:

            logger.warning(
                f"OHLC anomaly - {rule}: "
                f"{count} rows"
            )

    # ========================================================
    # 6. Future trade date
    # ========================================================

    today = pd.Timestamp.today().normalize()

    future_date_mask = (
        dataframe["trade_date"] > today
    )

    if future_date_mask.any():

        count = future_date_mask.sum()

        errors.append(
            f"trade_date in the future: "
            f"{count} rows"
        )

    # ========================================================
    # 7. Final result
    # ========================================================

    if errors:

        raise ValueError(
            "Quality check failed:\n- "
            + "\n- ".join(errors)
        )

    logger.info("Quality check passed")

    return dataframe


# ============================================================
# FINAL CURATED TRANSFORMATION
# ============================================================

def transform_to_curated(dataframe):
    """
    Complete RAW → CURATED transformation.
    """

    logger.info("Starting RAW → CURATED transformation")

    # 1. Validate source contract
    dataframe = validate_source_schema(dataframe)

    # 2. Standardize column names
    dataframe = rename_columns(dataframe)

    # 3. Select only curated columns
    #
    # This intentionally removes source-only fields such as:
    # No, persen, percentage, or future unexpected fields.
    dataframe = dataframe[EXPECTED_CURATED_COLUMNS].copy()

    # 4. Cast data types
    dataframe = cast_datatypes(dataframe)

    # 5. Quality checks
    dataframe = quality_check(dataframe)

    logger.info(
        f"RAW → CURATED completed. "
        f"Rows: {len(dataframe):,}, "
        f"Columns: {len(dataframe.columns)}"
    )

    return dataframe


# ============================================================
# TEST
# ============================================================

# Example:
#
# start_date = datetime.strptime(
#     "2026-08-30",
#     "%Y-%m-%d"
# ).date()
#
# end_date = datetime.strptime(
#     "2026-09-01",
#     "%Y-%m-%d"
# ).date()
#
# dfs = load_raw(start_date, end_date)
#
# df_raw = dfs[
#     "raw/idx/year=2026/month=08/data.parquet"
# ]
#
# df_curated = transform_to_curated(df_raw)
#
# print(df_curated.head())
# print(df_curated.dtypes)



In [5]:
# start_date = datetime.strptime(
#     "2026-08-30",
#     "%Y-%m-%d"
# ).date()
#
# end_date = datetime.strptime(
#     "2026-09-01",
#     "%Y-%m-%d"
# ).date()
#
# dfs = load_raw(start_date, end_date)
#
df_raw = dfs[
    "raw/idx/year=2026/month=08/data.parquet"
]
#
df_curated = transform_to_curated(df_raw)
#
print(df_curated.head())
print(df_curated.dtypes)

NameError: name 'dfs' is not defined

In [ ]:
def init_dim_stock(start_date, end_date):
    months = get_month_list(start_date, end_date)
    s3_client = boto3.client("s3")
    df = []

    for i, year_month in enumerate(months):
        month, year = year_month % 100, year_month // 100
        mm, yyyy = f"{month:02d}", f"{year:04d}"

        if i != len(months) - 1:
            key = f"curated/idx/year={yyyy}/month={mm}/data.parquet"
            logger.info(f"Loading curated: {key}")
            df.append(read_df_from_s3(BUCKET_NAME, key, s3_client))
        else:
            first_date = date(year, month, 1) if len(months) > 1 else start_date
            for current_date in pd.date_range(first_date, end_date):
                current_date = current_date.date()
                key = f"curated/idx/year={yyyy}/month={mm}/daily/date={current_date}/data.parquet"
                logger.info(f"Loading curated: {key}")
                data = read_df_from_s3(BUCKET_NAME, key, s3_client)
                if data is not None:
                    df.append(data)
                else:
                    logger.info(f"No data found: {key}")

    df = pd.concat(df, ignore_index=True)

    dim_stock = (
        df[["trade_date", "stock_code", "stock_name"]]
        .sort_values(["stock_code", "trade_date"])
        .reset_index(drop=True)
    )

    dim_stock["name_changed"] = (
        dim_stock.groupby("stock_code")["stock_name"]
        .transform(lambda x: x.ne(x.shift()))
    )

    dim_stock["version"] = (
        dim_stock.groupby("stock_code")["name_changed"]
        .cumsum()
    )

    dim_stock = (
        dim_stock
        .groupby(["stock_code", "version"], as_index=False)
        .agg(
            stock_name=("stock_name", "first"),
            valid_from=("trade_date", "first"),
            valid_to=("trade_date", "last")
        )
    )

    dim_stock["is_active"] = (
        dim_stock["valid_to"]
        == dim_stock.groupby("stock_code")["valid_to"].transform("max")
    ).astype(int)

    dim_stock = dim_stock[
        [
            "stock_code",
            "stock_name",
            "valid_from",
            "valid_to",
            "is_active"
        ]
    ]

    active_idx = (
        dim_stock.groupby("stock_code")["valid_to"]
        .idxmax()
    )

    dim_stock.loc[active_idx, "valid_to"] = pd.NaT

    return dim_stock

In [8]:
df = rename_columns(df_check)
df

,No,stock_summary_id,trade_date,stock_code,stock_name,remarks,previous_price,open_price,first_trade_price,high_price,...,tradeable_shares,index_weight,foreign_sell,foreign_buy,delisting_date,non_regular_volume,non_regular_value,non_regular_frequency,persen,percentage
0,1,4065341,2026-08-03T00:00:00,AADI,Adaro Andalan Indonesia Tbk.,--MO1SD0F10000A121------------,9225.0,9300.0,9300.0,9300.0,...,7.786892e+09,1.498977e+09,2396100.0,503600.0,,147964.0,1.339833e+09,16.0,None,None
1,2,4065342,2026-08-03T00:00:00,AALI,Astra Agro Lestari Tbk.,--MO113E100000D232------------,6875.0,6900.0,6900.0,7175.0,...,1.924688e+09,3.903268e+08,1111300.0,738100.0,,0.0,0.000000e+00,0.0,None,None
2,3,4065343,2026-08-03T00:00:00,ABBA,Mahaka Media Tbk.,--U-4100000000E614-E---------X,30.0,0.0,0.0,0.0,...,3.935893e+09,1.298451e+09,0.0,0.0,,0.0,0.000000e+00,0.0,None,None
3,4,4065344,2026-08-03T00:00:00,ABDA,Asuransi Bina Dana Arta Tbk.,--UO2105000000G412------------,3500.0,0.0,0.0,3530.0,...,6.208067e+08,5.947328e+07,0.0,0.0,,0.0,0.000000e+00,0.0,None,None
4,5,4065345,2026-08-03T00:00:00,ABMM,ABM Investama Tbk.,--MO1135000000A121------------,2510.0,2550.0,2550.0,2580.0,...,2.753165e+09,4.129748e+08,14600.0,197600.0,,16200.0,4.357800e+07,1.0,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18292,959,4083633,2026-08-31T00:00:00,ZATA,Bersama Zatta Jaya Tbk.,--UO2100000000E741-------Y----,67.0,67.0,67.0,69.0,...,1.700000e+09,2.208110e+09,7435000.0,8877000.0,,0.0,0.000000e+00,0.0,None,None
18293,960,4083634,2026-08-31T00:00:00,ZBRA,Dosni Roha Indonesia Tbk.,--U-4100000000C311B--L---Y---X,50.0,0.0,0.0,0.0,...,2.510706e+09,5.021413e+08,0.0,0.0,,0.0,0.000000e+00,0.0,None,None
18294,961,4083635,2026-08-31T00:00:00,ZINC,Kapuas Prima Coal Tbk.,--U-4100000000B146--D--------X,30.0,0.0,0.0,0.0,...,2.525000e+10,8.733975e+09,0.0,0.0,,0.0,0.000000e+00,0.0,None,None
18295,962,4083636,2026-08-31T00:00:00,ZONE,Mega Perintis Tbk.,--UO2130000000E741------------,780.0,0.0,0.0,780.0,...,8.701715e+08,1.325271e+08,0.0,6300.0,,0.0,0.000000e+00,0.0,None,None


In [321]:
new_code = con.execute("""
    select distinct StockCode 
    from
    read_parquet(
                's3://afdal-idx-stock-data-s3/raw/idx/year=2026/month=08/data.parquet'
            )
""").df()

In [327]:
import duckdb
import logging

logger = logging.getLogger(__name__)

BUCKET_NAME = "afdal-idx-stock-data-s3"
DIM_STOCK_KEY = "modeled/dim_stock/data.parquet"

def _get_duckdb_connection():
    con = duckdb.connect()
    con.execute("INSTALL httpfs;")
    con.execute("LOAD httpfs;")
    con.execute("""
        CREATE OR REPLACE SECRET (
            TYPE S3,
            PROVIDER credential_chain
        );
    """)
    return con

In [443]:
con.execute(
    f""" 
    select 
    --regexp_extract(d1.remarks, '00([A-Z][0-9]{{3}})', 1) as idx_ic_code1,
    --regexp_extract(remarks, '00([A-Z][0-9]{{3}})',1 ) AS idx_ic_code,
        *
    from read_parquet(
    --'s3://{BUCKET_NAME}/curated/idx/year=2020/month=01/data.parquet'
    's3://{BUCKET_NAME}/modeled/fact_stock_daily/year=2026/month=02/data.parquet' 
    ) as d1
    """
).df()

,date_key,stock_code,idx_ic_code,previous_price,open_price,first_trade_price,high_price,low_price,close_price,price_change,volume,trading_value,frequency,foreign_buy,foreign_sell,non_regular_volume,non_regular_value,non_regular_frequency,month,year
0,20260202,AADI,A121,7600.0,7975.0,7950.0,8150.0,7725.0,8075.0,475.0,51350200,4.087480e+11,21252,10612100,16862600,1039,7736510.0,8,02,2026
1,20260202,AALI,D232,7400.0,7400.0,7425.0,7500.0,7150.0,7275.0,-125.0,1250400,9.103860e+09,1224,556600,307600,0,0.0,0,02,2026
2,20260202,ABBA,E614,51.0,0.0,0.0,52.0,50.0,50.0,-1.0,645600,3.291570e+07,158,0,0,0,0.0,0,02,2026
3,20260202,ABDA,G412,2340.0,0.0,0.0,2340.0,2340.0,2340.0,0.0,1800,4.212000e+06,3,0,0,0,0.0,0,02,2026
4,20260202,ABMM,A121,2880.0,2880.0,2900.0,2900.0,2780.0,2800.0,-80.0,1594600,4.501914e+09,950,308800,445900,0,0.0,0,02,2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17239,20260227,ZATA,E741,163.0,0.0,0.0,0.0,0.0,163.0,0.0,0,0.000000e+00,0,0,0,0,0.0,0,02,2026
17240,20260227,ZBRA,C311,50.0,0.0,0.0,0.0,0.0,50.0,0.0,0,0.000000e+00,0,0,0,0,0.0,0,02,2026
17241,20260227,ZINC,B146,37.0,0.0,0.0,37.0,37.0,37.0,0.0,6569700,2.430789e+08,258,1626300,0,0,0.0,0,02,2026
17242,20260227,ZONE,E741,474.0,486.0,482.0,486.0,474.0,482.0,8.0,59400,2.842540e+07,33,0,2200,0,0.0,0,02,2026
